# Who Said What: audio → transcript with speaker roles

Uses **NVIDIA Nemotron 3 Diarization** (who spoke when) + **Parakeet TDT 0.6B v3** (speech → text).

1. `Runtime → Change runtime type → T4 GPU`
2. Run all cells
3. Open the `gradio.live` link and upload your audio

In [ ]:
!apt-get -qq install -y ffmpeg > /dev/null
!pip -q install "nemo-toolkit[asr] @ git+https://github.com/NVIDIA/NeMo.git@cf724ac337d1ebc7d0dda1e23fb80916f52927a5" gradio soundfile librosa "python-telegram-bot>=21" "anthropic>=1.8"

In [ ]:
!git clone -q https://github.com/glitchg/glitchg.git
%cd glitchg/tools/audio-diarization

## Option A: web UI (upload / record)

In [ ]:
!python app.py --share

## Option B: one file from Python

In [ ]:
from google.colab import files
from diarize_transcribe.pipeline import run
from diarize_transcribe.formatters import render

uploaded = files.upload()
path = next(iter(uploaded))
result = run(path)
print(render('txt', result.turns, roles=['Manager', 'Client']))

## Option C: Telegram bot

1. Click the 🔑 **Secrets** icon in the left sidebar and add:
   - `TELEGRAM_BOT_TOKEN`: your token from @BotFather
   - `ALLOWED_USER_IDS`: your Telegram user id (send `/start` to the bot to see it)
   - `ANTHROPIC_API_KEY` *(optional)*: enables automatic roles
2. Switch **Notebook access** on for each secret, then run the cell below.
3. Forward a voice message to the bot.

The bot stops when the Colab session ends. Free Colab sessions are disconnected after a period of inactivity and have a maximum runtime. For a bot that is always on, use the Docker image (see the README).

In [ ]:
import os
from google.colab import userdata

for key in ('TELEGRAM_BOT_TOKEN', 'ALLOWED_USER_IDS', 'ANTHROPIC_API_KEY'):
    try:
        os.environ[key] = userdata.get(key)
    except Exception:
        print(f'{key} not set (skipped)')

!python bot.py